In [1]:
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import XinferenceEmbeddings

# from utils import get_qwen_vl_7B_models

In [2]:
def get_qwen2_vl_models():
    llm = ChatOpenAI(temperature=0.2, 
                     top_p=0.2,
                     model="/gemini/code/neb_assistant/models/qwen/Qwen2-VL-7B-Instruct",
                     api_key="XXXX",
                     base_url="http://localhost:8000/v1")
    
    chat = ChatOpenAI(temperature=0.2, 
                      top_p=0.2,
                      model="/gemini/code/neb_assistant/models/qwen/Qwen2-VL-7B-Instruct",
                      api_key="XXXX",
                      base_url="http://localhost:8000/v1")
    
    embed = XinferenceEmbeddings(model_uid="bge-m3",
                                 server_url="http://direct.virtaicloud.com:40202")

    return llm, chat, embed
    
llm, chat, _ = get_qwen2_vl_models()

/root/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-12 21:45:07,896	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
chat(messages="Q:你是谁？A:")
# llm("Q:你是谁？A:")

/tmp/ipykernel_1387/2412742181.py:1: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use invoke instead.
  chat(messages="Q:你是谁？A:")


AIMessage(content='我是来自阿里云的大规模语言模型，我叫通义千问。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 26, 'total_tokens': 43}, 'model_name': '/gemini/code/neb_assistant/models/qwen/Qwen2-VL-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-a71867fb-f1b2-4e2e-8e7d-573817bb57ab-0', usage_metadata={'input_tokens': 26, 'output_tokens': 17, 'total_tokens': 43})

In [11]:
import os
import base64
from mimetypes import guess_type
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate

# Function to encode a local image into data URL 
def local_image_to_data_url(image_path):
    mime_type, _ = guess_type(image_path)
    # Default to png
    if mime_type is None:
        mime_type = 'image/png'

    # Read and encode the image file
    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(image_file.read()).decode('utf-8')

    # Construct the data URL
    return f"data:{mime_type};base64,{base64_encoded_data}"


prompt_template = HumanMessagePromptTemplate.from_template(
    template=[
                {"type": "text", "text": "描述图片的内容"},
                {"type": "image_url", "image_url": "{encoded_image_url}"}
             ]
)

summarize_image_prompt = ChatPromptTemplate.from_messages([prompt_template])
image_chain = summarize_image_prompt | chat 

print(os.getcwd())

# img_file = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
img_file = "assets/hurricane.jpeg"
page3_encoded = local_image_to_data_url(img_file)

/gemini/code/neb_assistant


In [5]:
response = image_chain.invoke(input={"encoded_image_url":page3_encoded})
print(response)

content='这张图片是一个雷达反射率的气象图，显示了在海口地区观测到的风暴系统。图中有许多颜色标记，表示不同的反射率强度，从绿色（低反射率）到黄色和红色（高反射率）。\n\n图中的主要特征是一个明显的风暴中心，中心部分的反射率非常高，呈红色和黄色，显示出天气系统中的强烈降水活动。风暴的边缘区域呈现出绿色和蓝色，表示反射率较低，通常与较轻的降水或无降水区域对应。\n\n地图的右上角提供了雷达站的信息和观测时间，即2024年9月6日22时08分38秒。地图的右下角还有一个标注，显示图像是由中国气象局雷达气象中心提供的。\n\n总体来看，这张图是展示一个热带风暴或飓风的雷达反射率分布情况，帮助气象学家了解风暴的强度和结构。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 188, 'prompt_tokens': 382, 'total_tokens': 570}, 'model_name': '/gemini/code/neb_assistant/models/qwen/Qwen2-VL-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-37502925-a64c-49ba-91b9-06d30f640ae7-0' usage_metadata={'input_tokens': 382, 'output_tokens': 188, 'total_tokens': 570}


In [12]:
prompt_template = HumanMessagePromptTemplate.from_template(
    template=[
                {"type": "text", "text": "描述图片的内容"},
                {"type": "image_url", "image_url": "{encoded_image_url}"}
             ]
)

prompt_template.format(encoded_image_url="1234")

HumanMessage(content=[{'type': 'text', 'text': '描述图片的内容'}, {'type': 'image_url', 'image_url': {'url': '1234'}}])